# Vegetation Filter — Random Forest Pipeline

Multi-scale geometric eigenfeatures (pgeof) + Random Forest classifier.
Sections: data loading → feature demo → NaN audit → training → inference → feature importance.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import joblib

from io_utils import load_las, get_scalar_field, get_labels, write_las_with_prediction
from feature_extraction import extract_features, spatial_subsample
from train import collect_training_data, train_model, save_model
from predict import predict_las, propagate_to_full_cloud

# ── Config ────────────────────────────────────────────────────────────────────
RADII          = [0.5, 1.0, 2.0]
SCALAR_FIELDS  = ["intensity"]   # add "return_number", "red", etc. if available
LABEL_FIELD    = "Classification"
UNLABELED_VAL  = -1
CORE_VOXEL     = 0.1
TRAIN_LAS      = "data/labeled_train.las"   # adjust paths
INFER_LAS      = "data/infer.las"
MODEL_PATH     = "models/rf_veg.joblib"

## 1. Data Loading Sanity Check

In [ ]:
xyz, las = load_las(TRAIN_LAS)
labels = get_labels(las, LABEL_FIELD)

print(f"Points: {len(xyz):,}  |  XYZ range: {xyz.min(0)} – {xyz.max(0)}")
print(f"\nAvailable dimensions: {list(las.point_format.dimension_names)}")
extra = [d.name for d in las.point_format.extra_dims]
if extra:
    print(f"Extra dims: {extra}")

print("\nLabel distribution:")
classes, counts = np.unique(labels, return_counts=True)
for cls, cnt in zip(classes, counts):
    tag = "(unlabeled)" if cls == UNLABELED_VAL else ""
    print(f"  {cls:4d} {tag}: {cnt:8,}  ({100*cnt/len(labels):.1f}%)")

## 2. Single-Scale Feature Extraction Demo (bounding-box crop)

In [ ]:
# Crop a 10 m box around the point-cloud median for a quick demo
demo_center = np.median(xyz, axis=0)
half = 5.0
mask = np.all((xyz >= demo_center - half) & (xyz <= demo_center + half), axis=1)
xyz_crop = xyz[mask]
print(f"Crop: {len(xyz_crop):,} points")

# Build scalar field dict for the crop
demo_scalar_fields = {}
for fname in SCALAR_FIELDS:
    try:
        field_vals = get_scalar_field(las, fname)
        demo_scalar_fields[fname] = field_vals[mask]
    except KeyError:
        pass

# Extract at a single scale for speed
demo_radius = [1.0]
X_demo = extract_features(
    xyz=xyz_crop,
    xyz_search=xyz_crop,
    radii=demo_radius,
    scalar_fields=demo_scalar_fields,
)
print(f"Feature matrix: {X_demo.shape}")
print(X_demo.describe().T[["mean", "std", "min", "max"]].round(4))

## 3. NaN Audit

In [ ]:
nan_frac = np.isnan(X_demo.values).mean(axis=0)
nan_series = pd.Series(nan_frac, index=X_demo.columns).sort_values(ascending=False)

print("Columns with any NaN (top 20):")
print(nan_series[nan_series > 0].head(20).to_string())

fig, ax = plt.subplots(figsize=(12, 3))
ax.bar(range(len(nan_series)), nan_series.values)
ax.set_xticks(range(len(nan_series)))
ax.set_xticklabels(nan_series.index, rotation=90, fontsize=7)
ax.set_ylabel("NaN fraction")
ax.set_title("NaN rate per feature (SimpleImputer will fill these)")
plt.tight_layout()
plt.show()

## 4. Training

In [ ]:
import os
os.makedirs("models", exist_ok=True)

X_train, y_train = collect_training_data(
    las_paths=[TRAIN_LAS],
    radii=RADII,
    scalar_field_names=SCALAR_FIELDS,
    label_field=LABEL_FIELD,
    unlabeled_value=UNLABELED_VAL,
    core_voxel_size=CORE_VOXEL,
)

pipeline = train_model(X_train, y_train)
save_model(pipeline, list(X_train.columns), RADII, SCALAR_FIELDS, MODEL_PATH)

print(f"\nFeature count: {len(X_train.columns)}")
print(f"OOB score: {pipeline.named_steps['rf'].oob_score_:.4f}")

## 5. Inference + Predicted-Class Visualization

In [ ]:
OUTPUT_LAS = "data/infer_predicted.las"

predict_las(
    las_path=INFER_LAS,
    model_path=MODEL_PATH,
    output_path=OUTPUT_LAS,
    core_voxel_size=CORE_VOXEL,
    write_probabilities=True,
)

# Visualise on a thin horizontal slice
xyz_inf, las_inf = load_las(OUTPUT_LAS)
from io_utils import get_scalar_field as gsf
pred = gsf(las_inf, "PredictedClass").astype(int)
prob = gsf(las_inf, "VegProbability")

# Slice around median Z
z_med = np.median(xyz_inf[:, 2])
slc = np.abs(xyz_inf[:, 2] - z_med) < 0.5

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
sc0 = axes[0].scatter(xyz_inf[slc, 0], xyz_inf[slc, 1],
                       c=pred[slc], cmap="tab10", s=0.5, linewidths=0)
axes[0].set_title("Predicted class (horizontal slice ±0.5 m)")
plt.colorbar(sc0, ax=axes[0], label="class")

sc1 = axes[1].scatter(xyz_inf[slc, 0], xyz_inf[slc, 1],
                       c=prob[slc], cmap="RdYlGn", vmin=0, vmax=1, s=0.5, linewidths=0)
axes[1].set_title("Veg probability")
plt.colorbar(sc1, ax=axes[1], label="P(veg)")

for ax in axes:
    ax.set_aspect("equal")
    ax.set_xlabel("X"); ax.set_ylabel("Y")
plt.tight_layout()
plt.show()

## 6. Feature Importance (grouped by scale)

In [ ]:
bundle = joblib.load(MODEL_PATH)
rf = bundle["pipeline"].named_steps["rf"]
feat_names = bundle["feature_names"]
importances = rf.feature_importances_

imp_df = pd.DataFrame({"feature": feat_names, "importance": importances})

# Extract scale tag from feature name (last token after _r)
def _scale(name):
    parts = name.split("_r")
    return f"r{parts[-1]}" if len(parts) > 1 else "unknown"

imp_df["scale"] = imp_df["feature"].apply(_scale)
imp_df["base"] = imp_df["feature"].apply(lambda n: n.rsplit("_r", 1)[0])

scales = sorted(imp_df["scale"].unique())
n_scales = len(scales)
colors = cm.tab10(np.linspace(0, 1, n_scales))

fig, axes = plt.subplots(1, n_scales, figsize=(5 * n_scales, 6), sharey=False)
if n_scales == 1:
    axes = [axes]

for ax, scale, color in zip(axes, scales, colors):
    sub = imp_df[imp_df["scale"] == scale].sort_values("importance", ascending=True)
    ax.barh(sub["base"], sub["importance"], color=color)
    ax.set_title(f"Scale {scale}")
    ax.set_xlabel("Importance")

plt.suptitle("Random Forest Feature Importances by Scale", fontsize=13)
plt.tight_layout()
plt.show()

print("\nTop 15 features overall:")
print(imp_df.nlargest(15, "importance")[["feature", "importance"]].to_string(index=False))

---
## Legacy — CloudCompare/cloudComPy Pipeline

The original unsupervised Bonneau-2019-style filter (Gaussian curvature − 3D density).
Kept for reference; requires `cloudComPy` in the environment.

In [ ]:
# import cloudComPy as ccp
# import numpy as np
# data = '/data/Scans/E/Alignments/E_20210924_aligned.bin'
# fullcloud = ccp.loadPointCloud(data)
# xyz = fullcloud.toNpArray()
# median_point = np.median(xyz, axis=0)
# curv_radii = np.arange(0.1,0.5,0.2)
# dens3d_radii = np.arange(0.5,0.15,-0.2)
# assert len(curv_radii) == len(dens3d_radii)
#
# curv = ccp.CurvatureType.GAUSSIAN_CURV
# dens3d = ccp.Density.DENSITY_3D
#
# test_cloud = ccp.ccPointCloud()
# test_point = [-18,22,1787]
# box_size = 5
# box_min = np.array(test_point) - box_size/2
# box_max = np.array(test_point) + box_size/2
# box_mask = np.all((xyz >= box_min) & (xyz <= box_max), axis=1)
# box_points = xyz[box_mask]
# test_cloud.coordsFromNPArray_copy(box_points)
# refCloud = ccp.CloudSamplingTools.resampleCloudSpatially(test_cloud, 0.01)
# (spatialCloud, res) = test_cloud.partialClone(refCloud)
# test_cloud = spatialCloud
#
# for sr, dr in zip(curv_radii, dens3d_radii):
#     ccp.computeLocalDensity(dens3d, dr, [test_cloud])
#     ccp.computeCurvature(curv, sr, [test_cloud])
#
# names = test_cloud.getScalarFieldDic()
# curv_names = sorted([n for n in names if 'curvature' in n],
#                     key=lambda x: float(x.split('(')[1].split(')')[0]), reverse=True)
# dens_names = sorted([n for n in names if 'density' in n],
#                     key=lambda x: float(x.split('=')[1].split(')')[0]))
#
# ratios = []
# for cn, dn in zip(curv_names, dens_names):
#     curv_values = test_cloud.getScalarField(names[cn]).toNpArray()
#     dens_values = test_cloud.getScalarField(names[dn]).toNpArray()
#     curv_values = (curv_values - np.nanmin(curv_values)) / (np.nanmax(curv_values) - np.nanmin(curv_values))
#     dens_values = (dens_values - np.nanmin(dens_values)) / (np.nanmax(dens_values) - np.nanmin(dens_values))
#     ratios.append(curv_values - dens_values)
#
# total = np.sum(ratios, axis=0)
# test_cloud.addScalarField('curv_density_ratio')
# sf = test_cloud.getScalarField('curv_density_ratio')
# sf.fromNpArrayCopy(total)
# names = test_cloud.getScalarFieldDic()
# test_cloud.sfBilateralFilter(names['curv_density_ratio'])
# test_cloud.setCurrentDisplayedScalarField(-1)
# ccp.SavePointCloud(test_cloud, 'test_output.bin')